# Demanda industrial regionalizada (AccumulatedAnnualDemand)

Calcula la demanda regional multiplicando los valores nacionales de `AccumulatedAnnualDemand` del SAND base (`SAND_Nacional_base/01-04-2026 SAND BASE v10.xlsx`) por la participación porcentual regional leída **directamente de la hoja `Participacion_Fuel`** de `Insumos/Participacion_Regional_Industrial.xlsx` (columnas `Fuel`, `Anio` y una columna por región).

La salida replica **exactamente el formato SAND del archivo de entrada**: hoja `Parameters`, columna `Parameter`, las mismas columnas índice (incluidas las vacías: `TECHNOLOGY`, `EMISSION`, `MODE_OF_OPERATION`, `TIMESLICE`, `STORAGE`, `REGION2`, `Time indipendent variables`), los años 2022–2055 y la columna indicadora `Tiene datos (≠0)`. Los códigos `FUEL` llevan el prefijo regional: `AN_INDCLIM`, `SE_INDDHT`, etc.

Los códigos `INDOTH_COA` / `INDOTH_ELC` (uso `Otros`) no están en el diccionario de mapeo y quedan excluidos.

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', 200)

In [ ]:
# --- Configuración ---
RUTA_NACIONAL = "SAND_Nacional_base/01-04-2026 SAND BASE v10.xlsx"
RUTA_PORCENTAJES = "Insumos/Participacion_Regional_Industrial.xlsx"
HOJA_PORCENTAJES = "Participacion_Fuel"
RUTA_SALIDA = "Insumos/AccumulatedAnnualDemand_Regional_Industrial.xlsx"

PARAMETRO = "AccumulatedAnnualDemand"

# Códigos del archivo Nacional (con prefijo sectorial IND) -> texto de la hoja Participacion_Fuel
mapeo_fuel = {
    "INDCLIM": "Aire acondicionado",
    "INDDHT": "Calor directo",
    "INDIHT": "Calor indirecto",
    "INDMPW": "Fuerza motriz",
    "INDILU": "Iluminacion",   # sin tilde, tal como viene en el Excel
    "INDREF": "Refrigeracion", # sin tilde, tal como viene en el Excel
}

# Región (columna de la hoja Participacion_Fuel) -> prefijo del modelo regional
# Prefijos verificados contra CSV_Regional/FUEL.csv; 'Este' corresponde a la región SE.
mapeo_region = {
    "Antioquia": "AN",
    "Caribe": "CA",
    "Este": "SE",
    "Insular": "IN",
    "Nordeste": "NE",
    "Oriente": "OR",
    "Suroccidente": "SO",
}

## 1. Porcentajes regionales desde `Participacion_Fuel`

La hoja viene en formato pivote (`Fuel`, `Anio`, una columna por región). Se pasa a formato largo: `Region`, `FUEL_Texto`, `Año`, `Porcentaje`. Nulos → 0.

In [ ]:
df_pct_raw = pd.read_excel(RUTA_PORCENTAJES, sheet_name=HOJA_PORCENTAJES)
print("Columnas de origen:", df_pct_raw.columns.tolist())

cols_region = [c for c in df_pct_raw.columns if c not in ("Fuel", "Anio")]
df_porcentajes = df_pct_raw.melt(
    id_vars=["Fuel", "Anio"], value_vars=cols_region,
    var_name="Region", value_name="Porcentaje",
)
df_porcentajes = df_porcentajes.rename(columns={"Fuel": "FUEL_Texto", "Anio": "Año"})
df_porcentajes["Porcentaje"] = pd.to_numeric(df_porcentajes["Porcentaje"], errors="coerce").fillna(0)

print("Textos de FUEL:", sorted(df_porcentajes["FUEL_Texto"].unique()))
print("Regiones:", sorted(df_porcentajes["Region"].unique()))
print("Años:", df_porcentajes["Año"].min(), "-", df_porcentajes["Año"].max())
df_porcentajes.head(10)

In [ ]:
# Verificaciones: participación por FUEL_Texto-Año debe sumar ~1 (o 0) y los textos
# del diccionario deben existir en la hoja
chk = df_porcentajes.groupby(["FUEL_Texto", "Año"])["Porcentaje"].sum()
malas = chk[((chk - 1).abs() > 1e-6) & (chk.abs() > 1e-6)]
print(f"Grupos FUEL-año con suma != 1 (y != 0): {len(malas)} de {len(chk)}")

faltantes = set(mapeo_fuel.values()) - set(df_porcentajes["FUEL_Texto"].unique())
print("Textos del mapeo sin porcentajes:", faltantes or "ninguno")

## 2. Lectura y filtro del Dataframe Nacional

Se filtra `Parameter == AccumulatedAnnualDemand`, se conservan solo los códigos `FUEL` presentes en `mapeo_fuel` y se pasa a formato largo (`melt`): `Codigo_FUEL`, `Año`, `Valor_Nacional`. Se guarda la estructura de columnas del SAND para replicarla en la salida.

In [ ]:
df_nacional_raw = pd.read_excel(RUTA_NACIONAL, sheet_name=0)

# Estructura exacta del SAND de entrada (se replica en la salida)
COLUMNAS_SAND = df_nacional_raw.columns.tolist()
ANIOS_SAND = [c for c in COLUMNAS_SAND if str(c).strip().isdigit()]
print("Años del SAND:", ANIOS_SAND[0], "-", ANIOS_SAND[-1])

df_nac = df_nacional_raw[df_nacional_raw["Parameter"] == PARAMETRO].copy()
df_nac["FUEL"] = df_nac["FUEL"].astype(str).str.strip()
df_nac = df_nac[df_nac["FUEL"].isin(mapeo_fuel.keys())]
print("Códigos FUEL nacionales retenidos:", sorted(df_nac["FUEL"].unique()))

REGION_NACIONAL = df_nac["REGION"].dropna().iloc[0]
print("REGION:", REGION_NACIONAL)

df_nac = df_nac.melt(id_vars="FUEL", value_vars=ANIOS_SAND,
                     var_name="Año", value_name="Valor_Nacional")
df_nac = df_nac.rename(columns={"FUEL": "Codigo_FUEL"})
df_nac["Año"] = df_nac["Año"].astype(str).str.strip().astype(int)
df_nac["Valor_Nacional"] = pd.to_numeric(df_nac["Valor_Nacional"], errors="coerce").fillna(0)

# Nombre descriptivo para el cruce
df_nac["FUEL_Texto"] = df_nac["Codigo_FUEL"].map(mapeo_fuel)
print(df_nac.shape)
df_nac.head()

## 3. Años interconectados

Si el Nacional llega más lejos que los porcentajes (ej. 2055 vs 2054), se extiende la participación del último año disponible hacia los años faltantes, para no dejar demanda nacional sin repartir.

In [ ]:
anios_nac = set(df_nac["Año"].unique())
anios_pct = set(df_porcentajes["Año"].unique())
faltan = sorted(anios_nac - anios_pct)

if faltan:
    ultimo = max(anios_pct)
    print(f"Años del Nacional sin porcentajes: {faltan} -> se usa la participación de {ultimo}")
    base = df_porcentajes[df_porcentajes["Año"] == ultimo]
    extension = pd.concat(
        [base.assign(**{"Año": a}) for a in faltan], ignore_index=True
    )
    df_porcentajes = pd.concat([df_porcentajes, extension], ignore_index=True)
else:
    print("Los porcentajes cubren todos los años del Nacional.")

sobran = sorted(anios_pct - anios_nac)
if sobran:
    print(f"Años de porcentajes sin dato nacional (se ignoran en el cruce): {sobran}")

## 4. Cruce y cálculo del valor regional

`merge` por `FUEL_Texto` y `Año`; `Valor_Regional = Valor_Nacional * Porcentaje`. Los `NaN` se llenan con 0.

In [ ]:
df_regional = df_nac.merge(df_porcentajes, on=["FUEL_Texto", "Año"], how="left")

sin_cruce = df_regional["Region"].isna().sum()
if sin_cruce:
    print(f"Advertencia: {sin_cruce} filas nacionales sin porcentaje regional (quedan en 0)")

df_regional["Porcentaje"] = df_regional["Porcentaje"].fillna(0)
df_regional["Valor_Regional"] = (df_regional["Valor_Nacional"] * df_regional["Porcentaje"]).fillna(0)

df_regional.head(10)

In [ ]:
# Verificación: la suma de las regiones debe reproducir el valor nacional
chk = (df_regional.groupby(["Codigo_FUEL", "Año"])
       .agg(Nacional=("Valor_Nacional", "first"), Suma_Regiones=("Valor_Regional", "sum"))
       .reset_index())
chk["Diferencia"] = (chk["Nacional"] - chk["Suma_Regiones"]).abs()
malas = chk[(chk["Diferencia"] > 1e-9 * chk["Nacional"].abs().clip(lower=1)) & (chk["Nacional"] != 0)]
print(f"Combinaciones código-año donde la suma regional no reproduce el nacional: {len(malas)} de {len(chk)}")
if len(malas):
    print(malas.head(10).to_string(index=False))

## 5. Salida en formato SAND

`FUEL = <prefijo región>_<código nacional>` (ej. `AN_INDCLIM`, `SE_INDDHT`). Se construye una tabla con **exactamente las mismas columnas del SAND de entrada** (incluidas las vacías), hoja `Parameters`. La columna `Tiene datos (≠0)` se recalcula: 1 si la fila tiene algún valor distinto de cero, 0 si no.

In [ ]:
df_regional["Prefijo"] = df_regional["Region"].map(mapeo_region)

sin_prefijo = df_regional["Prefijo"].isna() & df_regional["Region"].notna()
if sin_prefijo.any():
    raise ValueError(f"Regiones sin prefijo definido: {sorted(df_regional.loc[sin_prefijo, 'Region'].unique())}")

df_calc = df_regional[df_regional["Region"].notna()].copy()
df_calc["FUEL"] = df_calc["Prefijo"] + "_" + df_calc["Codigo_FUEL"]

# Pivote a formato ancho: una fila por FUEL regional, una columna por año
wide = df_calc.pivot_table(index="FUEL", columns="Año", values="Valor_Regional", fill_value=0)
wide = wide.sort_index()

# Encabezados de año tal como vienen en el SAND (texto '2022', '2023', ...)
mapa_anio = {int(str(c).strip()): c for c in ANIOS_SAND}
wide.columns = [mapa_anio[a] for a in wide.columns]

# Armar la tabla final con las columnas exactas del SAND de entrada
df_salida = pd.DataFrame(index=wide.index)
for col in COLUMNAS_SAND:
    if col == "Parameter":
        df_salida[col] = PARAMETRO
    elif col == "REGION":
        df_salida[col] = REGION_NACIONAL
    elif col == "FUEL":
        df_salida[col] = wide.index
    elif col in wide.columns:
        df_salida[col] = wide[col]
    elif str(col).startswith("Tiene datos"):
        df_salida[col] = (wide != 0).any(axis=1).astype(int)
    else:
        df_salida[col] = pd.NA  # columnas índice sin uso: quedan vacías

df_salida = df_salida.reset_index(drop=True)
print("Columnas de salida idénticas al SAND de entrada:", df_salida.columns.tolist() == COLUMNAS_SAND)
print("FUELs generados:", df_salida["FUEL"].nunique())
df_salida.head(15)

In [ ]:
with pd.ExcelWriter(RUTA_SALIDA, engine="openpyxl") as writer:
    df_salida.to_excel(writer, sheet_name="Parameters", index=False)

print(f"Archivo exportado en: {RUTA_SALIDA}")
print(f"Filas: {len(df_salida)} | Columnas: {len(df_salida.columns)}")